In [3]:
import pandas as pd
import sqlite3

csv_path = 'g31_Equipment_Operators_Final_2.csv'
db_path = 'MB_&_FIC.db'

# Carregar o dataset
df = pd.read_csv(csv_path)
conn = sqlite3.connect(db_path)

# 1. Tabela Equipment
equipment = df[['equipment_id', 'name', 'creation_date', 'equipment_type']].drop_duplicates()
equipment = equipment.rename(columns={'equipment_id': 'id', 'equipment_type': 'type'})
equipment.to_sql('Equipment', conn, if_exists='replace', index=False)

# 2. Tabela Operator
operators = df[['operator_id', 'title', 'category', 'birth_date']].drop_duplicates()
operators = operators.rename(columns={'operator_id': 'id'})
operators.to_sql('Operator', conn, if_exists='replace', index=False)

# 3. Tabela Maintenance_type
m_types = df[['maintenance_type_id', 'equipment_id']].dropna().drop_duplicates()
m_types = m_types.rename(columns={'maintenance_type_id': 'id'})
m_types['id'] = m_types['id'].astype(int)
m_types.to_sql('Maintenance_type', conn, if_exists='replace', index=False)

# 4. Tabela Maintenance_event
events = df[['maintenance_event_id', 'equipment_id', 'maintenance_type_id', 'maintenance_date', 'extra_info', 'maintenance_date_final']].dropna(subset=['maintenance_event_id'])
events = events.rename(columns={'maintenance_event_id': 'id'})
for col in ['id', 'equipment_id', 'maintenance_type_id']:
    events[col] = events[col].astype(int)
events.to_sql('Maintenance_event', conn, if_exists='replace', index=False)

# 5. Tabela Equipment_operator
usage = df[['equipment_id', 'operator_id', 'utilization_date', 'cost']]
usage.index = usage.index + 1
usage.to_sql('Equipment_operator', conn, if_exists='replace', index=True, index_label='id')


cursor = conn.cursor()
print("Tabelas criadas com sucesso:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
print([row[0] for row in cursor.fetchall()])

conn.commit()
conn.close()

Tabelas criadas com sucesso:
['Equipment', 'Operator', 'Maintenance_type', 'Maintenance_event', 'Equipment_operator']
